# Stacje aktywne przez cały wybrany rok

W pierwszej komórce ustaw `YEAR` (domyślnie `2024`; można zmienić na `2025`). Wypożyczenia i ocena ciągłości dotyczą okresu od 1 stycznia wybranego roku włącznie do 1 stycznia następnego roku wyłącznie.

Kryteria aktywności pozostają takie jak w oryginale: status instalacji, data instalacji i ciągłość historycznych rekordów. Dane są wczytywane z tego samego pliku `Data/rides_data_without_discontinuities.parquet` oraz przez `london_cycles.stations()`. Wyszukiwanie katalogu projektu w katalogach nadrzędnych działa również z nowej lokalizacji notebooka.


In [ ]:
# Analizowany rok
YEAR = 2024

from pathlib import Path
import pandas as pd
from london_cycles import stations as load_stations


# Ścieżki do danych
def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start)

    for path in [current, *current.parents]:
        if (path / "Data").exists():
            return path

    raise FileNotFoundError("Could not find project root containing Data/")


PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "Data"
DATA_FILE = DATA_DIR / "rides_data_without_discontinuities.parquet"


# Wczytanie danych o wypożyczeniach
rides_data = pd.read_parquet(DATA_FILE, columns=[
    "start_station_id",
    "start_station_name",
    "start_date",
    "start_day_of_week"
])

# Wczytanie historycznych danych stacji z Hugging Face
stations_df = pd.DataFrame(load_stations())


# Analizowany okres: od początku wybranego roku do początku kolejnego roku
start = pd.Timestamp(year=YEAR, month=1, day=1)
end = pd.Timestamp(year=YEAR + 1, month=1, day=1)

# Wypożyczenia wyłącznie z wybranego roku: początek włącznie, koniec wyłącznie
rides_data["start_date"] = pd.to_datetime(rides_data["start_date"])
rides_data = rides_data[
    (rides_data["start_date"] >= start) &
    (rides_data["start_date"] < end)
].copy()

# Kopia danych historycznych
stations = stations_df.copy()

# Konwersja dat
for col in ["valid_from", "valid_to", "install_date"]:
    stations[col] = pd.to_datetime(stations[col], errors="coerce")

# Ujednolicenie identyfikatorów
stations["terminal_id"] = pd.to_numeric(
    stations["terminal_name"], errors="coerce"
)

# Zachowanie rekordów stacji zainstalowanych najpóźniej na początku wybranego roku
stations = stations[
    (stations["installed"] == True) &
    (stations["install_date"].isna() | (stations["install_date"] <= start))
].copy()

# Brak końca ważności oznacza otwarty przedział
stations["valid_to"] = stations["valid_to"].fillna(end)


# Sprawdzenie ciągłości historycznych rekordów
def active_full_period(group):
    covered_until = start

    for row in group.sort_values("valid_from").itertuples():

        if pd.isna(row.valid_from) or row.valid_from > covered_until:
            return False

        covered_until = max(covered_until, row.valid_to)

        if covered_until >= end:
            return True

    return False


# Identyfikatory stacji spełniających warunek
active_station_ids = {
    int(station_id)
    for station_id, group in stations.groupby("terminal_id")
    if active_full_period(group)
}

# Identyfikatory stacji z naszych wypożyczeń
trip_ids = set(
    pd.to_numeric(
        rides_data["start_station_id"], errors="coerce"
    ).dropna().astype(int)
)

# Porównanie zbiorów
valid_stations = trip_ids & active_station_ids
excluded_stations = trip_ids - active_station_ids


# Podsumowanie
print(f"ANALIZA STACJI: {YEAR}")
print("-" * 40)
print(f"Stacje w wypożyczeniach: {len(trip_ids)}")
print(f"Stacje z ciągłym statusem instalacji: {len(active_station_ids)}")
print(f"Stacje zachowane: {len(valid_stations)}")
print(f"Stacje odrzucone: {len(excluded_stations)}")
print(f"Procent zachowanych stacji: {len(valid_stations) / len(trip_ids) * 100:.2f}%")


# Lista odrzuconych stacji
excluded_summary = (
    rides_data[
        pd.to_numeric(
            rides_data["start_station_id"], errors="coerce"
        ).isin(excluded_stations)
    ]
    .groupby(["start_station_id", "start_station_name"])
    .agg(
        rental_count=("start_date", "size")
    )
    .reset_index()
    .sort_values("rental_count", ascending=False)
)

display(excluded_summary)

# Historia odrzuconych stacji

Poniższe tabele pokazują pełną historię stacji odrzuconych w analizie wybranego roku, jako kontekst oceny ciągłości. Liczby stacji i wypożyczeń w podsumowaniu dotyczą wyłącznie wybranego roku.


In [ ]:
# Historyczne informacje o odrzuconych stacjach
excluded_details = stations_df.copy()

# Ujednolicenie identyfikatorów
excluded_details["terminal_id"] = pd.to_numeric(
    excluded_details["terminal_name"], errors="coerce"
)

# Wybór odrzuconych stacji
excluded_details = excluded_details[
    excluded_details["terminal_id"].isin(excluded_stations)
].copy()

# Wyświetlenie historii stacji
display(
    excluded_details[
        [
            "terminal_id",
            "common_name",
            "installed",
            "install_date",
            "removal_date",
            "valid_from",
            "valid_to"
        ]
    ].sort_values(["terminal_id", "valid_from"])
)

In [ ]:
# Odrzucone stacje w wybranym roku
ids_to_check = sorted(excluded_stations)

# Kopia danych
station_history = stations_df.copy()

# Identyfikatory terminali
station_history["terminal_id"] = pd.to_numeric(
    station_history["terminal_name"], errors="coerce"
)

# Historia odrzuconych stacji
display(
    station_history[
        station_history["terminal_id"].isin(ids_to_check)
    ][
        [
            "terminal_id",
            "place_id",
            "common_name",
            "lat",
            "lon",
            "installed",
            "valid_from",
            "valid_to"
        ]
    ].sort_values(["terminal_id", "valid_from"])
)

# Sprawdzenie wypożyczeń w historycznych okresach

Zachowano okresy diagnostyczne z oryginału. Daty opisują konkretne historyczne zdarzenia, dlatego nie są przesuwane na inny rok. Każdy okres jest ograniczany do granic `YEAR`; okresy bez części wspólnej z wybranym rokiem są pomijane. Te sprawdzenia nie dodają kryteriów wykluczania stacji.


In [ ]:
# Przygotowanie danych
rides_test = rides_data.copy()

rides_test["start_date"] = pd.to_datetime(rides_test["start_date"])
rides_test["start_station_id"] = pd.to_numeric(
    rides_test["start_station_id"], errors="coerce"
)

# Okresy do sprawdzenia
periods_to_check = [
    ("Camley Street", 22181, "2024-01-01", "2024-06-12"),
    ("Kennington Oval", 200032, "2024-05-21", "2024-06-03"),
    ("Brandon Street", 300060, "2025-01-07", "2025-01-17"),
    ("Newton Street", 300074, "2025-04-11", "2025-05-08"),
    ("Abyssinia Close", 300099, "2024-01-01", "2025-10-29"),
    ("Adeline Place", 300099, "2025-10-29", "2026-01-01")
]

# Sprawdzenie wypożyczeń
results = []

for name, station_id, start_date, end_date in periods_to_check:

    # Ograniczenie historycznego okresu do wybranego roku
    start_date = max(pd.Timestamp(start_date), start)
    end_date = min(pd.Timestamp(end_date), end)

    if start_date >= end_date:
        continue

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start_date) &
        (rides_test["start_date"] < end_date)
    ]

    results.append({
        "station": name,
        "period": f"{start_date:%Y-%m-%d} – {end_date:%Y-%m-%d}",
        "rental_count": len(subset),
        "station_names": ", ".join(
            subset["start_station_name"].dropna().unique()
        )
    })

display(pd.DataFrame(results))

In [ ]:
# Stacje i okresy do sprawdzenia
periods_to_check = {
    "Brandon Street": (300060, "2025-01-07", "2025-01-17"),
    "Newton Street": (300074, "2025-04-11", "2025-05-08"),
    "Kennington Oval": (200032, "2024-05-20", "2024-06-03")
}

# Dzienne wypożyczenia w podejrzanych okresach
for name, (station_id, start_date, end_date) in periods_to_check.items():

    # Ograniczenie historycznego okresu do wybranego roku
    start_date = max(pd.Timestamp(start_date), start)
    end_date = min(pd.Timestamp(end_date), end)

    if start_date >= end_date:
        continue

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start_date) &
        (rides_test["start_date"] < end_date)
    ].copy()

    daily_counts = (
        subset.groupby(subset["start_date"].dt.date)
        .size()
    )

    print(f"\n{name}")
    print(daily_counts.to_string())

In [ ]:
# Okresy do sprawdzenia
periods = {
    "Kennington Oval": (200032, "2024-05-10", "2024-06-15"),
    "Camley Street": (22181, "2024-01-01", "2024-06-20")
}

for name, (station_id, start_date, end_date) in periods.items():

    # Ograniczenie historycznego okresu do wybranego roku
    start_date = max(pd.Timestamp(start_date), start)
    end_date = min(pd.Timestamp(end_date), end)

    if start_date >= end_date:
        continue

    subset = rides_test[
        (rides_test["start_station_id"] == station_id) &
        (rides_test["start_date"] >= start_date) &
        (rides_test["start_date"] < end_date)
    ]

    daily_counts = (
        subset.groupby(subset["start_date"].dt.normalize())
        .size()
    )

    # Uwzględnienie dni bez wypożyczeń
    days = pd.date_range(start_date, end_date, inclusive="left")
    daily_counts = daily_counts.reindex(days, fill_value=0)

    print(f"\n{name}")
    print(f"Dni z wypożyczeniami: {(daily_counts > 0).sum()}")
    print(f"Dni bez wypożyczeń: {(daily_counts == 0).sum()}")

    display(daily_counts.to_frame("rental_count").T)